In [1]:
"""
PASSO 3 — DataProcess: normalização min-max e criação dos DatasetNew.

Para cada {Code}_TechIndi.csv, aplica normalização min-max em cada feature
(colunas 2 a 45), mantém o Label inalterado, e filtra dados a partir de
2017-01-01. Salva como {Code}_DatasetNew.csv.

Replica _IM_DataProcess.R:
    - Features = TechData[, -c(1, n)]   → colunas exceto Date e Label
    - Normalize = apply(Features, 2, f)  → min-max por coluna
    - data2 = Data1["2017-01-01/"]       → dados a partir de 2017

CORREÇÃO: trata NaN provenientes de indicadores técnicos com variância
zero ou valores ausentes. O R usa na.omit() para remover linhas com NaN;
aqui fazemos o mesmo e preenchemos eventuais NaN remanescentes com 0.

Uso:
    python 03_data_process.py
"""

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

# ===================== CONFIGURAÇÃO =====================
BASE_DIR = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICS")
SEC_NAMES = BASE_DIR / ".NewB3_pruned.csv"
DATE_START = "2017-01-01"
# ========================================================


def minmax_col(s: pd.Series) -> pd.Series:
    """
    Normalização min-max por coluna.
    Equivalente à função f(x) = (x - min(x)) / (max(x) - min(x)) do R.
    Retorna 0.0 quando max == min (variância zero) para evitar NaN.
    """
    mn = s.min(skipna=True)
    mx = s.max(skipna=True)
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(0.0, index=s.index)
    return (s - mn) / (mx - mn)


def main():
    codes_df = pd.read_csv(SEC_NAMES, dtype=str, encoding="utf-8-sig")
    codes = codes_df["Code"].str.strip().str.upper().tolist()

    print(f"Total de tickers: {len(codes)}")
    print(f"Filtro temporal: a partir de {DATE_START}")
    print(f"Modo: SOBRESCREVE arquivos existentes\n")

    report = []
    for code in tqdm(codes, desc="DataProcess"):
        infile = BASE_DIR / f"{code}_TechIndi.csv"
        outfile = BASE_DIR / f"{code}_DatasetNew.csv"

        if not infile.exists():
            report.append({"Code": code, "status": "no_TechIndi", "rows": 0})
            continue

        try:
            df = pd.read_csv(infile, encoding="utf-8-sig")

            # Identificar colunas: Date (1ª), Features (2ª a penúltima), Label (última)
            date_col = df.columns[0]
            label_col = df.columns[-1]
            feature_cols = df.columns[1:-1].tolist()

            # Converter data
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
            df = df.dropna(subset=[date_col]).sort_values(date_col).reset_index(drop=True)

            # ============ TRATAMENTO DE NaN ============
            # 1. Remover linhas onde TODAS as features são NaN
            features_raw = df[feature_cols].astype(float)
            mask_all_nan = features_raw.isna().all(axis=1)
            df = df[~mask_all_nan].reset_index(drop=True)

            # 2. Remover linhas com NaN no Label
            df = df.dropna(subset=[label_col]).reset_index(drop=True)

            # 3. Contar NaN restantes antes da normalização
            nan_before = df[feature_cols].astype(float).isna().sum().sum()

            # 4. Preencher NaN restantes nas features com 0
            df[feature_cols] = df[feature_cols].astype(float).fillna(0.0)
            # ==========================================

            # Normalizar features (min-max por coluna, sobre TODO o dataset)
            features = df[feature_cols].astype(float)
            normalized = features.apply(minmax_col, axis=0)

            # ============ GARANTIA FINAL ============
            nan_after = normalized.isna().sum().sum()
            if nan_after > 0:
                normalized = normalized.fillna(0.0)
            # ========================================

            # Montar DataFrame final
            out = pd.DataFrame()
            out[date_col] = df[date_col]
            for col in feature_cols:
                out[col] = normalized[col].values
            out[label_col] = df[label_col].values.astype(int)

            # Filtrar a partir de 2017-01-01
            out = out[out[date_col] >= pd.to_datetime(DATE_START)].reset_index(drop=True)

            if len(out) < 100:
                report.append({"Code": code, "status": f"too_few_rows ({len(out)})", "rows": len(out)})
                continue

            # Formatar data como string
            out[date_col] = out[date_col].dt.strftime("%Y-%m-%d")

            # Salvar (sobrescreve se já existe)
            out.to_csv(outfile, index=False, encoding="utf-8-sig")

            status = "ok" if nan_before == 0 else f"ok (corrigidos {nan_before} NaN)"
            report.append({"Code": code, "status": status, "rows": len(out)})

        except Exception as e:
            report.append({"Code": code, "status": f"error: {e}", "rows": 0})

    # Relatório
    report_df = pd.DataFrame(report)
    report_df.to_csv(BASE_DIR / "dataprocess_report.csv", index=False, encoding="utf-8-sig")

    n_ok = report_df["status"].str.startswith("ok").sum()
    n_nan_fixed = report_df["status"].str.contains("NaN").sum()
    n_fail = len(report_df) - n_ok
    avg_rows = report_df.loc[report_df["status"].str.startswith("ok"), "rows"].mean()

    print(f"\n{'='*50}")
    print(f"Concluído: {n_ok} gerados, {n_fail} falharam.")
    print(f"Ações com NaN corrigidos: {n_nan_fixed}")
    print(f"Média de linhas por DatasetNew: {avg_rows:.0f}")
    print(f"Colunas: Date + 44 features normalizadas + Label = 46")
    print(f"Relatório: dataprocess_report.csv")


if __name__ == "__main__":
    main()

Total de tickers: 246
Filtro temporal: a partir de 2017-01-01
Modo: SOBRESCREVE arquivos existentes



DataProcess: 100%|██████████| 246/246 [00:31<00:00,  7.72it/s]


Concluído: 246 gerados, 0 falharam.
Ações com NaN corrigidos: 0
Média de linhas por DatasetNew: 803
Colunas: Date + 44 features normalizadas + Label = 46
Relatório: dataprocess_report.csv
